# CPEN 455 2026W1: Cache on a Budget, T4 quickstart

Runs on a free T4: **Google Colab** (Runtime → Change runtime type → T4 GPU) or **AWS SageMaker Studio Lab** (GPU runtime). Kaggle (GPU T4) works the same way.

The setup script installs the locked course dependencies in `.venv/`. Use this T4 for development and report measurements, and record your device and dtype. Every command below runs through `.venv/bin/python` after `source scripts/env.sh`, never the notebook's own Python.

**Sessions are temporary.** Commit and push your `student/` folder often (or work from a Google Drive copy).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 1. Get your copy of the repository and set up the environment (≈ 3–5 min)

In [ ]:
import os
REPO = "https://github.com/<your-username>/CPEN455-Project-2026W1.git"   # your private copy of DSL-Lab/CPEN455-Project-2026W1
if not os.path.exists("CPEN455-Project-2026W1"):
    !git clone {REPO}
%cd CPEN455-Project-2026W1
%pip install -q uv
!bash scripts/setup_t4.sh

## 2. Public unit tests
Most tests fail in the starter code until you implement the TODOs; `tests/test_harness.py` passes from the start.

In [ ]:
!source scripts/env.sh && .venv/bin/python -m pytest tests -q

## 3. Part 0: time vanilla nanoGPT generation

In [ ]:
%%bash
source scripts/env.sh
.venv/bin/python - <<'PY'
import time, torch
from cpen455kv.weights import load_model
from cpen455kv.data import load_split
from cpen455kv.timing import fixed_sequences
m = load_model("checkpoints/gpt2-medium.pt", "cuda", torch.float16)
idx = fixed_sequences(load_split("cbt-val", limit=1), 1, 64).cuda()
m.generate(idx, 8, top_k=1); torch.cuda.synchronize()
t = time.time(); m.generate(idx, 256, top_k=1); torch.cuda.synchronize()
print(f"vanilla nanoGPT: {256 / (time.time() - t):.1f} tokens/s")
PY

## 4. Part 2.5: your cache vs vanilla nanoGPT (after Part 2)

In [ ]:
!source scripts/env.sh && .venv/bin/python -m cpen455kv.part2_bench --strict-env --batches 1 16 --new 512 --out student/part2_results.json

## 5. Quick quality check of a policy (no timing)
Change `CPEN455_POLICY` to `StreamingLLM`, `H2O` or `MyPolicy`; develop on `cbt-val` (Children's Book Test) and keep `cbt-test` for the validation run below.

In [ ]:
!source scripts/env.sh && CPEN455_POLICY=MyPolicy .venv/bin/python -m cpen455kv.grade --split cbt-val --limit 64 --skip-timing --out run_quick.json

## 6. Full validation run
This runs the complete `cbt-test` split and writes the `validation.json` you submit. Freeze your settings before running it. Software, hardware and checkpoint metadata are recorded automatically.

In [ ]:
!source scripts/env.sh && .venv/bin/python -m cpen455kv.grade --strict-env --split cbt-test --out validation.json

## 7. Pre-submission check

In [ ]:
!source scripts/env.sh && .venv/bin/python -m cpen455kv.check .